# CLASIFICACIÓN CON UNA RED NEURONAL CONVOLUCIONAL

Se hará uso de pytorch para la implementación de la red neuronal convolucional.

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import pandas as pd
from torch.utils.data import Dataset
import torch.nn.init as init
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import numpy as np
import os
import random
from PIL import Image
from torchvision import models, transforms
import re

In [2]:
####DATASET PRUEBA

##Se crea una dataframe pandas con dos columnas, la primera la ruta de la imagen y la segunda o 0 o 1 random
# Obtén la lista de rutas de imágenes en un directorio específico
directorio_imagenes = './NIFTI PREPROC'
rutas_imagenes = [os.path.join(directorio_imagenes, imagen) for imagen in os.listdir(directorio_imagenes)]

rutas_imagenes = np.array(rutas_imagenes)
labels = np.zeros_like(rutas_imagenes)

#Se obtienen las etiquetas
df = pd.read_csv('etiquetas.txt', delimiter='\t')
df = df.dropna(how="all")

# Recorrer fila a fila
for index, row in df.iterrows():
    first = row["Pac"]
    second = row["Centro"]
    label = row["Etiqueta"]
    patron = re.compile(rf'.*(?:{first}.*{second}|{first}.*{second}).*')

    indices_coincidentes = [i for i, s in enumerate(rutas_imagenes) if re.match(patron, s)]

    for i in indices_coincidentes:
        labels[i] = label
    
    if len(indices_coincidentes) < 3:
        print("AVISO")
        print(indices_coincidentes)
        rutas_coincidentes = [rutas_imagenes[i] for i in indices_coincidentes]
        print(rutas_coincidentes)

labels = labels.astype("float")
# Crea el DataFrame con las dos columnas
df_train = pd.DataFrame({'imgs': rutas_imagenes, 'label': labels})

# Muestra el DataFrame
# print(df_train)

# Eliminar las filas donde label es igual a 2
df_train = df_train.drop(df_train[df_train['label'] == 2.0].index)
# print(df_train)

##Se comprueba si alguna de la rutas sigue en el dataset
for ruta in rutas_coincidentes:
    print(df_train[df_train['imgs'] == ruta])

AVISO
[300]
['./NIFTI PREPROC\\wA05_P46_HVN.nii']
                                 imgs  label
300  ./NIFTI PREPROC\wA05_P46_HVN.nii    0.0


In [3]:
######COMPARACION ESTRUCRURALES

patron = re.compile(r'.*(CT_|RMT1_|RMT2_)')
# Seleccionar los strings que contengan "ct" o "rmt" y quitar los elementos anteriores a la ocurrencia
seleccionados = list(set([re.sub(patron, r'\1', s)  for s in rutas_imagenes if re.search(patron, s)]))

# Imprimir los resultados
print(seleccionados)

rutas_imagenes_no_prep = [imagen for imagen in os.listdir('./NIFTI NO PREPROC')]
patron = re.compile(r'^(CT_|RMT1_|RMT2_)')
seleccionados_no_prep = [re.sub(patron, r'\1', s) for s in rutas_imagenes_no_prep if re.search(patron, s)]

print(seleccionados_no_prep)

elementos_no_comunes = [elemento for elemento in seleccionados_no_prep if elemento not in seleccionados]

print(len(elementos_no_comunes))
print(len(seleccionados_no_prep)-len(seleccionados))

print(elementos_no_comunes)

['CT_P21_HVN.nii', 'CT_P27_HVN.nii', 'CT_P64_H9O.nii', 'CT_P32_HVN.nii', 'CT_P48_HVN.nii', 'CT_P53_H9O.nii', 'CT_P70_HVN.nii', 'RMT1_P25_HVN.nii', 'CT_P50_H9O.nii', 'CT_P61_HVN.nii', 'CT_P72_H9O.nii', 'CT_P30_HVN.nii', 'CT_P51_HVN.nii', 'CT_P62_H9O.nii', 'CT_P06_HVN.nii', 'CT_P15_HVN.nii', 'RMT1_P17_HVN.nii', 'RMT1_P34_HVN.nii', 'RMT1_P66_HVN.nii', 'CT_P42_HVN.nii', 'RMT1_P51_HVN.nii', 'CT_P56_HVN.nii', 'RMT1_P42_HVN.nii', 'CT_P75_H9O.nii', 'CT_P52_H9O.nii', 'CT_P68_HVN.nii', 'CT_P14_HVN.nii', 'CT_P36_HVN.nii', 'RMT1_P48_HVN.nii', 'RMT1_P70_HVN.nii', 'CT_P54_H9O.nii', 'CT_P23_HVN.nii', 'RMT1_P18_HVN.nii', 'RMT1_P12_HVN.nii', 'CT_P70_H9O.nii', 'CT_P20_HVN.nii', 'CT_P55_H9O.nii', 'CT_P40_HVN.nii', 'CT_P04_HVN.nii', 'CT_P37_HVN.nii', 'RMT1_P71_HVN.nii', 'CT_P69_H9O.nii', 'CT_P58_HVN.nii', 'CT_P54_HVN.nii', 'CT_P41_HVN.nii', 'CT_P61_H9O.nii', 'CT_P63_HVN.nii', 'RMT1_P27_HVN.nii', 'CT_P31_HVN.nii', 'RMT1_P69_HVN.nii', 'CT_P19_HVN.nii', 'RMT1_P13_HVN.nii', 'CT_P47_HVN.nii', 'RMT1_P65_HVN.nii

In [4]:
##comparacion amiloide
patron = re.compile(r'.*(A)')
# Seleccionar los strings que contengan "ct" o "rmt" y quitar los elementos anteriores a la ocurrencia
seleccionados = [re.sub(patron, r'\1', s) for s in rutas_imagenes if re.search(patron, s)]

# Imprimir los resultados
print(seleccionados)

rutas_imagenes_no_prep = [imagen for imagen in os.listdir('./NIFTI NO PREPROC')]

seleccionados_no_prep = [re.sub(patron, r'\1', s) for s in rutas_imagenes_no_prep if re.search(patron, s)]

print(seleccionados_no_prep)

elementos_no_comunes2 = [elemento for elemento in seleccionados_no_prep if elemento not in seleccionados]

print(elementos_no_comunes2)

['A05_P05_HVN.nii', 'A05_P06_HVN.nii', 'A05_P07_HVN.nii', 'A05_P08_HVN.nii', 'A05_P09_HVN.nii', 'A05_P10_HVN.nii', 'A05_P11_HVN.nii', 'A05_P12_HVN.nii', 'A05_P13_HVN.nii', 'A05_P15_HVN.nii', 'A05_P16_HVN.nii', 'A05_P17_HVN.nii', 'A05_P18_HVN.nii', 'A05_P19_HVN.nii', 'A05_P20_HVN.nii', 'A05_P21_HVN.nii', 'A05_P22_HVN.nii', 'A05_P23_HVN.nii', 'A05_P24_HVN.nii', 'A05_P25_HVN.nii', 'A05_P26_HVN.nii', 'A05_P27_HVN.nii', 'A05_P29_HVN.nii', 'A05_P30_HVN.nii', 'A05_P31_HVN.nii', 'A05_P32_HVN.nii', 'A05_P33_HVN.nii', 'A05_P34_HVN.nii', 'A05_P35_HVN.nii', 'A05_P36_HVN.nii', 'A05_P37_HVN.nii', 'A05_P38_HVN.nii', 'A05_P39_HVN.nii', 'A05_P40_HVN.nii', 'A05_P41_HVN.nii', 'A05_P42_HVN.nii', 'A05_P44_HVN.nii', 'A05_P45_HVN.nii', 'A05_P46_HVN.nii', 'A05_P48_HVN.nii', 'A05_P49_HVN.nii', 'A05_P50_HVN.nii', 'A05_P51_HVN.nii', 'A05_P52_HVN.nii', 'A05_P53_HVN.nii', 'A05_P54_HVN.nii', 'A05_P59_HVN.nii', 'A05_P60_HVN.nii', 'A05_P61_HVN.nii', 'A05_P63_HVN.nii', 'A05_P64_HVN.nii', 'A05_P66_HVN.nii', 'A05_P68_HV

In [5]:
#COMPARACION FDG
##comparacion amiloide
patron = re.compile(r'.*(FDG)')
# Seleccionar los strings que contengan "ct" o "rmt" y quitar los elementos anteriores a la ocurrencia
seleccionados = [re.sub(patron, r'\1', s) for s in rutas_imagenes if re.search(patron, s)]

# Imprimir los resultados
print(seleccionados)

rutas_imagenes_no_prep = [imagen for imagen in os.listdir('./NIFTI NO PREPROC')]

seleccionados_no_prep = [re.sub(patron, r'\1', s) for s in rutas_imagenes_no_prep if re.search(patron, s)]

print(seleccionados_no_prep)

elementos_no_comunes3 = [elemento for elemento in seleccionados_no_prep if elemento not in seleccionados]

print(elementos_no_comunes3)

['FDG_P02_HVN.nii', 'FDG_P08_HVN.nii', 'FDG_P10_HVN.nii', 'FDG_P12_HVN.nii', 'FDG_P13_HVN.nii', 'FDG_P15_HVN.nii', 'FDG_P17_HVN.nii', 'FDG_P19_HVN.nii', 'FDG_P20_HVN.nii', 'FDG_P21_HVN.nii', 'FDG_P22_HVN.nii', 'FDG_P24_HVN.nii', 'FDG_P25_HVN.nii', 'FDG_P26_HVN.nii', 'FDG_P27_HVN.nii', 'FDG_P34_HVN.nii', 'FDG_P47_HVN.nii', 'FDG_P48_HVN.nii', 'FDG_P50_H9O.nii', 'FDG_P51_H9O.nii', 'FDG_P52_H9O.nii', 'FDG_P53_H9O.nii', 'FDG_P54_H9O.nii', 'FDG_P55_H9O.nii', 'FDG_P56_H9O.nii', 'FDG_P57_H9O.nii', 'FDG_P58_H9O.nii', 'FDG_P59_H9O.nii', 'FDG_P59_HVN.nii', 'FDG_P60_H9O.nii', 'FDG_P60_HVN.nii', 'FDG_P61_H9O.nii', 'FDG_P62_H9O.nii', 'FDG_P63_H9O.nii', 'FDG_P64_H9O.nii', 'FDG_P65_H9O.nii', 'FDG_P66_H9O.nii', 'FDG_P66_HVN.nii', 'FDG_P67_H9O.nii', 'FDG_P68_H9O.nii', 'FDG_P69_H9O.nii', 'FDG_P69_HVN.nii', 'FDG_P70_H9O.nii', 'FDG_P71_H9O.nii', 'FDG_P71_HVN.nii', 'FDG_P72_H9O.nii', 'FDG_P73_H9O.nii', 'FDG_P74_H9O.nii', 'FDG_P75_H9O.nii']
['FDG_P02_HVN.nii', 'FDG_P08_HVN.nii', 'FDG_P10_HVN.nii', 'FDG_P12_H

In [6]:
import os
import shutil

# Directorio donde se encuentran los archivos originales
directorio_origen = './NIFTI NO PREPROC'

# Directorio donde se crearán los archivos seleccionados
directorio_destino = './IMAGENES_SEGMENTAR'

# String con los nombres de los archivos deseados
nombres_deseados = elementos_no_comunes
# Borrar el directorio de destino si ya existe
if os.path.exists(directorio_destino):
    shutil.rmtree(directorio_destino)

# Crear el directorio de destino
os.makedirs(directorio_destino)

# Iterar sobre los archivos en el directorio de origen
for archivo in os.listdir(directorio_origen):
    # Verificar si el nombre del archivo está en la lista de nombres deseados
    if archivo in nombres_deseados:
        # Construir la ruta completa de origen y destino
        ruta_origen = os.path.join(directorio_origen, archivo)
        ruta_destino = os.path.join(directorio_destino, archivo)
        # Mover el archivo al directorio de destino
        shutil.copy(ruta_origen, ruta_destino)

print("Archivos seleccionados han sido movidos a la nueva carpeta.")

Archivos seleccionados han sido movidos a la nueva carpeta.


In [7]:
def learning_curve(train_loss, val_loss):
    epochs = range(1, len(train_loss) + 1)

    plt.plot(epochs, np.array(train_loss), label='Training Loss')
    plt.plot(epochs, np.array(val_loss), label='Validation Loss')

    plt.title('Learning Curve')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.ylim(0, 1)
    plt.legend()
    plt.show()

In [8]:
# Verificar si CUDA está disponible
cuda_available = torch.cuda.is_available()

# Imprimir la versión de PyTorch y la disponibilidad de CUDA
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {cuda_available}")

# Imprimir el nombre de la GPU si CUDA está disponible
if cuda_available:
    print(f"GPU name: {torch.cuda.get_device_name(0)}")
    # Obtener la versión de CUDA
    cuda_version = torch.version.cuda
    print(f"CUDA version: {cuda_version}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

PyTorch version: 2.2.0
CUDA available: True
GPU name: NVIDIA GeForce RTX 2060
CUDA version: 12.1
Using device: cuda


In [9]:
####CREACION DATASET

import os
import pandas as pd

# Directorio que contiene los archivos
directorio = './IMGS_MEDIA'

# Obtener la lista de archivos en el directorio
archivos = os.listdir(directorio)

# Crear una lista para almacenar las rutas de los archivos y las etiquetas
data = []

# Iterar sobre los archivos
for archivo in archivos:
    # Obtener la ruta completa del archivo
    ruta_completa = os.path.join(directorio, archivo)
    # Obtener los elementos después de hacer split("_")
    elementos = archivo.split("_")
    # Obtener la etiqueta (cuarto elemento)
    etiqueta = elementos[3] if len(elementos) >= 4 else None
    # Agregar la ruta y la etiqueta a la lista de datos
    data.append((ruta_completa, int(etiqueta)))

# Crear el DataFrame de Pandas
df_mean = pd.DataFrame(data, columns=['Ruta', 'Etiqueta'])

# Mostrar el DataFrame
print(df_mean)

                            Ruta  Etiqueta
0   ./IMGS_MEDIA\_P02_HVN_1_.jpg         1
1   ./IMGS_MEDIA\_P08_HVN_1_.jpg         1
2   ./IMGS_MEDIA\_P10_HVN_1_.jpg         1
3   ./IMGS_MEDIA\_P12_HVN_0_.jpg         0
4   ./IMGS_MEDIA\_P15_HVN_0_.jpg         0
5   ./IMGS_MEDIA\_P17_HVN_0_.jpg         0
6   ./IMGS_MEDIA\_P19_HVN_0_.jpg         0
7   ./IMGS_MEDIA\_P20_HVN_0_.jpg         0
8   ./IMGS_MEDIA\_P21_HVN_1_.jpg         1
9   ./IMGS_MEDIA\_P22_HVN_1_.jpg         1
10  ./IMGS_MEDIA\_P24_HVN_1_.jpg         1
11  ./IMGS_MEDIA\_P25_HVN_0_.jpg         0
12  ./IMGS_MEDIA\_P26_HVN_0_.jpg         0
13  ./IMGS_MEDIA\_P27_HVN_1_.jpg         1
14  ./IMGS_MEDIA\_P34_HVN_0_.jpg         0
15  ./IMGS_MEDIA\_P47_HVN_0_.jpg         0
16  ./IMGS_MEDIA\_P48_HVN_0_.jpg         0
17  ./IMGS_MEDIA\_P50_H9O_1_.jpg         1
18  ./IMGS_MEDIA\_P51_H9O_0_.jpg         0
19  ./IMGS_MEDIA\_P52_H9O_1_.jpg         1
20  ./IMGS_MEDIA\_P53_H9O_0_.jpg         0
21  ./IMGS_MEDIA\_P54_H9O_1_.jpg         1
22  ./IMGS_

Entrenamos primero sin FineTune, haciendo uso de Adam, SGD y RMSProp. Se probaran varias topologías de redes para seleccionar las dos mejores y estudiar una posible mejora haciendo uso de Fine Tune. Se usarán batch sizes de 16, 8, 4, 1

Se creará una función para encapsular ls pruebas con resnet y otras redes. Se puede variar el lr del optimizador, no más parámetros, para una prueba más exhaustiva del optimizador, se recomienda hacerlo fuera de la función.

In [10]:
def accuracy(outputs, labels):
    preds = outputs > 0.5  # Si la salida es mayor que 0.5, consideramos que es positiva
    correct = (preds == labels.byte())  # Compara las predicciones con las etiquetas reales
    acc = correct.float().sum() / labels.size(0)  # Calcula la precisión
    return acc.item()

def recall(outputs, labels):
    preds = outputs > 0.5

    preds = preds.cpu()
    labels = labels.cpu()
    TP = sum((preds == 1) & (labels == 1))
    FN = sum((preds == 0) & (labels == 1))
    
    recall_score = TP / (TP + FN) if (TP + FN) > 0 else 0
    return recall_score

def precision(outputs, labels):
    preds = outputs > 0.5

    preds = preds.cpu()
    labels = labels.cpu()

    TP = sum((preds == 1) & (labels == 1))
    FP = sum((preds == 1) & (labels == 0))
    
    precision_score = TP / (TP + FP) if (TP + FP) > 0 else 0
    return precision_score

def prueba_k_fold_resnet_no_pre(dataset, num_folds, num_epochs, op, batch_size, lr, data_aug, ini):
    # Transformaciones para preprocesar las imágenes
    if data_aug:
        transform_t = transforms.Compose([
            transforms.Resize((224, 224)),
            # transforms.RandomResizedCrop(224),
            transforms.RandomHorizontalFlip(),
            # transforms.RandomRotation(45),
            # transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.2),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ])

        transform_v = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ])
    else:
        transform_t = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ])

        transform_v = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ])
    
    # Modelo personalizado basado en ResNet
    class CustomResNet(nn.Module):
        def __init__(self, num_classes_binary=1):
            super(CustomResNet, self).__init__()
            resnet = models.resnet18(weights=None)# New weights with accuracy 80.858%
            # Extraer las capas de la red ResNet, excluyendo la capa de clasificación
            self.features = nn.Sequential(*list(resnet.children())[:-1])
    
            # Personalizar la capa de salida para tus necesidades
            self.fc_binary = nn.Linear(512, num_classes_binary)
            
            if (ini== "G"):
                # Inicialización de Glorot para las capas fully connected
                self.apply(self.init_weights_glorot)
            elif(ini == "H"):
                self.apply(self.init_weights_he)

        def init_weights_glorot(self, m):
            if isinstance(m, nn.Linear):
                init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    init.constant_(m.bias, 0)

        def init_weights_he(self, m):
            if isinstance(m, nn.Linear):
                init.kaiming_uniform_(m.weight, mode='fan_in', nonlinearity='relu')
                if m.bias is not None:
                    init.constant_(m.bias, 0)
    
        def forward(self, x):
            x = self.features(x)
            x = x.view(x.size(0), -1)
    
    
            # Salida para la variable binaria
            out_binary = self.fc_binary(x)
    
            return out_binary
    
    # Definir una clase para el conjunto de datos personalizado
    ##adaptar esto para meter unas rutsa y obtener dataos y etiquetas
    class CustomDataset(Dataset):
        def __init__(self, dataframe, transform=None):
            self.data = dataframe
            self.transform = transform
    
        def __len__(self):
            return len(self.data)
    
        def __getitem__(self, idx):
            img_path = self.data.iloc[idx, 0]  # Ruta de la imagen en la primera columna
            image = Image.open(img_path).convert('RGB')
            # Etiquetas: una variable binaria
            label_binary = torch.tensor(self.data.iloc[idx, 1], dtype=torch.float32)
    
            if self.transform:
                image = self.transform(image)
    
            return image, label_binary
        
    #####5 fold CV
    from sklearn.model_selection import StratifiedKFold
    
    # Definir el número de folds
    criterion_binary = nn.BCEWithLogitsLoss()
    kf = StratifiedKFold(n_splits=num_folds, shuffle=True, random_state=33)
    
    # Listas para almacenar las pérdidas de entrenamiento y validación de cada fold
    train_losses_folds = []
    val_losses_folds = []
    
    train_accs_folds = []
    train_rec_folds =  []
    train_preci_folds = []

    val_accs_folds = []
    val_rec_folds =  []
    val_preci_folds = []
    # Iterar sobre los folds
    for fold, (train_index, val_index) in enumerate(kf.split(np.zeros(len(df_mean)),df_mean['Etiqueta'])):
        print(f"Fold {fold + 1}")
    
        # Dividir el conjunto de datos en entrenamiento y validación para este fold
        train_data_fold = dataset.iloc[train_index]
        val_data_fold = dataset.iloc[val_index]
    
        # Crear instancias del conjunto de datos personalizado para este fold
        train_dataset_fold = CustomDataset(dataframe=train_data_fold, transform=transform_t)
        val_dataset_fold = CustomDataset(dataframe=val_data_fold, transform=transform_v)
    
        # Crear DataLoader para cargar los datos en lotes
        train_loader_fold = DataLoader(dataset=train_dataset_fold, batch_size=batch_size, shuffle=True)
        val_loader_fold = DataLoader(dataset=val_dataset_fold, batch_size=batch_size, shuffle=False) 
    
        # Re-inicializar el modelo para cada fold
        model = CustomResNet(num_classes_binary=1).to(device)
        optimizer = op(model.parameters(), lr = lr)
        # Entrenamiento y validación para este fold
        train_losses_fold = []
        val_losses_fold = []
        train_accuracy_fold = []
        val_accuracy_fold = []
        train_recall_fold = []
        val_recall_fold = []
        train_preci_fold = []
        val_preci_fold = []

        for epoch in range(num_epochs):
            # Entrenamiento del modelo
            # Fase de entrenamiento
            model.train()
            loss_total_acumulada = 0.0
            lossT_bin_total = 0.0
            correct_train = 0
            recall_value_train = 0
            precision_value_train = 0
            for images, bin_label in train_loader_fold:
                # Reiniciar gradientes
                images, bin_label = images.to(device), bin_label.to(device)
        
                optimizer.zero_grad()
        
                # Forward pass
                bin_out = model(images)
        
                # Calcular pérdida para las variables continuas
                
        
        
                # Calcular pérdida para la variable binaria
                lossT_bin = criterion_binary(bin_out.squeeze(1), bin_label)
        
                # Pérdida total
                loss_total = lossT_bin
    
                loss_total_acumulada += loss_total
    
                # Backward pass y optimización
                loss_total.backward()
                optimizer.step()

                bin_pred = torch.sigmoid(bin_out.squeeze())
                correct_train += accuracy(bin_pred, bin_label)
                recall_value_train += recall(bin_pred, bin_label)
                precision_value_train += precision(bin_pred, bin_label)

        
    
            # Validación del modelo
            model.eval()
            val_loss = 0.0
            correct_valid = 0
            recall_value_valid = 0
            precision_value_valid = 0
            lossV_bin_total = 0.0
            with torch.no_grad():
                for images, bin_label in val_loader_fold:
                    images,  bin_label = images.to(device),  bin_label.to(device)
                    ##print(len(images)) aqui se hace con 32 imagenes
                    # Forward pass en el conjunto de validación
                    bin_out = model(images)
    
    
    
                    # Calcular pérdida para la variable binaria
                    lossV_bin = criterion_binary(bin_out.squeeze(1), bin_label)
    
                    # Sumar las pérdidas del conjunto de validación
                    val_loss += lossV_bin.item()

                    bin_pred = torch.sigmoid(bin_out.squeeze())
                    correct_valid += accuracy(bin_pred, bin_label)
                    recall_value_valid += recall(bin_pred, bin_label)
                    precision_value_valid += precision(bin_pred, bin_label)
                    
            
            
            avg_loss_total_acumulada = loss_total_acumulada / len(train_loader_fold)
            train_losses_fold.append(avg_loss_total_acumulada)
            acc_train = correct_train / len(train_loader_fold)
            recall_train = recall_value_train / len(train_loader_fold)
            precision_train = precision_value_train / len(train_loader_fold)
            train_accuracy_fold.append(acc_train)
            train_recall_fold.append(recall_train)
            train_preci_fold.append(precision_train)
        
            avg_val_loss = val_loss / len(val_loader_fold)
            val_losses_fold.append(avg_val_loss)
            acc_valid = correct_valid / len(val_loader_fold)
            recall_valid = recall_value_valid / len(val_loader_fold)
            precision_valid = precision_value_valid / len(val_loader_fold)
            val_accuracy_fold.append(acc_valid)
            val_recall_fold.append(recall_valid)
            val_preci_fold.append(precision_valid)
            # print(f'Epoch [{epoch+1}/{num_epochs}], Training Loss: {avg_loss_total_acumulada.item()}, Validation Loss: {avg_val_loss}')
        
        
        print(f"Accuracy Train: {np.mean(train_accuracy_fold)}. Accuracy valid: {np.mean(val_accuracy_fold)}")
        print(f"Recall Train: {np.mean(train_recall_fold)}. Recall valid: {np.mean(val_recall_fold)}")
        print(f"Precision Train: {np.mean(train_preci_fold)}. Precision valid: {np.mean(val_preci_fold)}")
        
        train_accs_folds.append(np.mean(train_accuracy_fold))
        train_rec_folds.append(np.mean(train_recall_fold))
        train_preci_folds.append(np.mean(train_preci_fold))

        val_accs_folds.append(np.mean(val_accuracy_fold))
        val_rec_folds.append(np.mean(val_recall_fold))
        val_preci_folds.append(np.mean(val_preci_fold))

        train_losses_folds.append(train_losses_fold)
        val_losses_folds.append(val_losses_fold)
        train_losses_fold = [i.cpu().detach().numpy() for i in train_losses_fold]
        learning_curve(train_losses_fold, val_losses_fold)
    
    train_losses_folds_cpu = []
    val_losses_folds_cpu = []
    for losses in val_losses_folds:
        val_losses_fold_cpu = []
        for loss in losses:
            val_losses_fold_cpu.append(loss)
        val_losses_folds_cpu.append(np.array(val_losses_fold_cpu))
    for losses in train_losses_folds:
        train_losses_fold_cpu = []
        for loss in losses:
            train_losses_fold_cpu.append(float(loss.cpu().detach().numpy()))
        train_losses_folds_cpu.append(np.array(train_losses_fold_cpu))
    print(len(train_losses_folds_cpu))
    avg_train = []
    avg_val = []
    for losses in train_losses_folds_cpu:
        avg_train.append(np.mean(losses))
    for losses in val_losses_folds_cpu:
        avg_val.append(np.mean(losses))
    
    # Imprimir pérdidas promedio para cada época
    for fold, (train_loss, val_loss) in enumerate(zip(avg_train, avg_val)):
        print(f"Fold {fold}: Training Loss: {train_loss}, Validation Loss: {val_loss}")
    
    print(f"TRAIN LOSS MEDIA: {np.mean(avg_train)} VALID LOSS MEDIA: {np.mean(avg_val)}")
    print(f"TRAIN ACC MEDIA: {np.mean(train_accs_folds)} VALID ACC MEDIA: {np.mean(val_accs_folds)}")        
    print(f"TRAIN RECALL MEDIA: {np.mean(train_rec_folds)} VALID RECALL MEDIA: {np.mean(val_rec_folds)}")
    print(f"TRAIN PRECISION MEDIA: {np.mean(train_preci_folds)} VALID PRECISION MEDIA: {np.mean(val_preci_folds)}")